# Pilot (Gemma only): throughput + tool-call error rate on Kaggle 2xT4
Research plan §11.3 (O1) / §11.6 / §11.8. Same protocol as pilot Version #1 (Granite 31.5 eps/h, Qwen 171.4 eps/h — DECISIONS D17): llama.cpp **b11344** `llama-server`, AgentDojo 0.1.35 **no-attack utility** on `banking` (n=16) + `slack` (n=21) = 37 episodes, `-c 24576 -np 4 --jinja --temp 0`, tensor-split `1,1` over both T4s.

**Model:** `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` → `gemma-4-26B_q4_0-it.gguf` (Google official QAT Q4_0, 14.44 GB, Apache-2.0, not gated). Gemma-only flags: `--swa-full` (plan §11.8, prefix reuse on sliding-window layers) and `--chat-template-kwargs {"enable_thinking":false}` (thinking off, plan §11.3).

**Decision rule (D20):** keep Gemma in the five if episodes/h ≥ 64, utility ≥ 20% and strict parse-error rate ≤ 5%.

**Terms**: self-hosted Apache-2.0 weights; fictional AgentDojo data; server on localhost only; no tunnels. Keep this notebook **private**. One Kaggle account.

Settings: Accelerator **GPU T4 x2**, Internet **on**. Then **Save Version → Save & Run All (Commit)**.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout, flush=True)
_n = subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.count('GPU ')
print(f'GPUs visible: {_n}', flush=True)
assert _n >= 2, 'Need Accelerator = GPU T4 x2 (tensor-split 1,1)'


## 1. Install pinned llama.cpp (CUDA 12.8) and agentdojo
b11344 (built 2 Oct 2026) post-dates llama.cpp's Gemma-4 support and the Gemma-4 tool-grammar fix (PR #29115, 19 Sep 2026).

In [ ]:
LLAMA_BUILD = 'b11344'
CUDA_TARBALL = 'https://github.com/ggml-org/llama.cpp/releases/download/b11344/llama-b11344-bin-ubuntu-cuda-12.8-x64.tar.gz'
CUDART_TARBALL = 'https://github.com/ggml-org/llama.cpp/releases/download/b11344/cudart-llama-b11344-bin-ubuntu-cuda-12.8-x64.tar.gz'
import os, subprocess
os.makedirs('/kaggle/working/llama', exist_ok=True)
for url in (CUDART_TARBALL, CUDA_TARBALL):
    fn = url.split('/')[-1]
    subprocess.run(['wget','-q','-O',f'/kaggle/working/llama/{fn}',url], check=True)
    subprocess.run(['tar','xzf',f'/kaggle/working/llama/{fn}','-C','/kaggle/working/llama'], check=True)
SERVER = '/kaggle/working/llama/build/bin/llama-server'
import glob
cands = glob.glob('/kaggle/working/llama/**/llama-server', recursive=True)
SERVER = cands[0]; print('llama-server at', SERVER)
print(subprocess.run([SERVER,'--version'],capture_output=True,text=True).stderr[:200])

In [ ]:
# pin agentdojo and deps
!pip -q install agentdojo==0.1.35 openai==3.23.0

## 2. Download the Gemma GGUF to scratch (D13)
* `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` → `gemma-4-26B_q4_0-it.gguf` (14,439,363,584 B ≈ 13.45 GiB), pinned revision `d1c082be9cf3`, SHA-256 `3eca3b8f6d7baf21…` checked below.
* Weights go to `/kaggle/tmp/models` if it exists, else `/tmp/models` — **never** `/kaggle/working/models` (~20 GB cap; ENOSPC in the first pilot). Any stale GGUF under `/kaggle/working/models` is removed first.
* Do **not** download the `*-mmproj.gguf` (vision projector; not needed for text agents).

In [ ]:
# Free /kaggle/working of any stale GGUFs / partial downloads (D13).
import os, shutil, subprocess
WORKING_MODELS = '/kaggle/working/models'
removed = []
if os.path.isdir(WORKING_MODELS):
    for root, dirs, files in os.walk(WORKING_MODELS, topdown=False):
        for name in files:
            if name.endswith(('.gguf', '.incomplete', '.lock')):
                fp = os.path.join(root, name)
                try: os.remove(fp); removed.append(fp)
                except FileNotFoundError: pass
print('removed from /kaggle/working/models:', removed or '(nothing)', flush=True)
print(subprocess.run(['df','-h','/kaggle/working','/tmp'],
                     capture_output=True, text=True).stdout, flush=True)


In [ ]:
import os, hashlib, shutil, subprocess, time
from huggingface_hub import hf_hub_download

for candidate in ('/kaggle/tmp/models', '/tmp/models'):
    parent = os.path.dirname(candidate)
    if os.path.isdir(parent) or parent == '/tmp':
        MODELS_DIR = candidate
        break
else:
    MODELS_DIR = '/tmp/models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['HF_HUB_CACHE'] = MODELS_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR
free = shutil.disk_usage(MODELS_DIR).free / (1024**3)
print(f'MODELS_DIR={MODELS_DIR}  free={free:.1f} GiB', flush=True)
assert free > 15.0, 'scratch too small for the 13.45 GiB Gemma GGUF'

GEMMA_REPO = 'google/gemma-4-26B-A4B-it-qat-q4_0-gguf'
GEMMA_REV = 'd1c082be9cf3c8a514acf63b8761f4b41935842e'
GEMMA_FILE = 'gemma-4-26B_q4_0-it.gguf'
GEMMA_SIZE = 14439363584
GEMMA_SHA256 = '3eca3b8f6d7baf218a7dd6bba5fb59a56ee25fe2d567b6f5f589b4f697eca51d'
t0 = time.time()
print(f'downloading {GEMMA_REPO}/{GEMMA_FILE} ...', flush=True)
GEMMA = hf_hub_download(GEMMA_REPO, GEMMA_FILE, revision=GEMMA_REV, local_dir=MODELS_DIR)
sz = os.path.getsize(GEMMA)
print(f'Gemma -> {GEMMA}  {sz/(1024**3):.2f} GiB  ({time.time()-t0:.0f}s)', flush=True)
assert sz == GEMMA_SIZE, f'size mismatch {sz} != {GEMMA_SIZE}'

def sha256(p, n=1<<22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while (b := f.read(n)): h.update(b)
    return h.hexdigest()
GEMMA_SHA = sha256(GEMMA)
print(f'sha256 {GEMMA_SHA}  match={GEMMA_SHA == GEMMA_SHA256}', flush=True)
assert GEMMA_SHA == GEMMA_SHA256, 'GGUF sha256 mismatch'


## 3. Serve Gemma (2×T4, tensor-split 1,1) and run no-attack utility
`serve()` tries the full plan flags first and logs which variant came up: (a) `--swa-full` + thinking-off kwargs, (b) without `--swa-full` (if the full SWA KV cache does not fit), (c) neither. Any fallback is printed as `SERVE_VARIANT=` and must be logged in DECISIONS.md. A background heartbeat prints every 5 min (elapsed, JSON traces written so far) so a long suite never looks hung (D16).

In [ ]:
import subprocess, time, os, requests, json
SERVE_VARIANTS = [
    ('swa_full+no_think', ['--swa-full', '--chat-template-kwargs', '{"enable_thinking":false}']),
    ('no_think',          ['--chat-template-kwargs', '{"enable_thinking":false}']),
    ('plain',             []),
]
SERVE_USED = None
def serve(model_path, ngl=99, ctx=24576, nslots=4, split='1,1', port=8099, timeout_s=600):
    global SERVE_USED
    for name, extra in SERVE_VARIANTS:
        cmd = [SERVER,'-m',model_path,'--host','127.0.0.1','--port',str(port),
               '-ngl',str(ngl),'-c',str(ctx),'-np',str(nslots),'--jinja','--temp','0']
        if split: cmd += ['--tensor-split', split]
        cmd += extra
        logf = f'/kaggle/working/server_{port}_{name}.log'
        print(f'serve[{name}]: ' + ' '.join(cmd[1:]), flush=True)
        p = subprocess.Popen(cmd, stdout=open(logf,'w'), stderr=subprocess.STDOUT)
        t0 = time.time()
        while time.time() - t0 < timeout_s:
            if p.poll() is not None:
                break
            try:
                if requests.get(f'http://127.0.0.1:{port}/v1/models', timeout=5).ok:
                    SERVE_USED = {'variant': name, 'cmd': cmd[1:], 'log': logf}
                    print(f'server up  SERVE_VARIANT={name}  ({time.time()-t0:.0f}s)', flush=True)
                    return p
            except Exception: pass
            time.sleep(3)
        p.kill(); p.wait()
        tail = open(logf).read()[-1500:]
        print(f'serve[{name}] FAILED (rc={p.returncode}); log tail:\n{tail}', flush=True)
    raise RuntimeError('llama-server did not start with any variant; see server_*.log')


In [ ]:
# attach this repo (add it as a Kaggle dataset or git clone) so attacks/scoring import
import sys; sys.path.insert(0, '/kaggle/working/code')  # adjust to where you put code/
import os; os.environ['LOCAL_LLM_PORT']='8099'
from agentdojo.task_suite.load_suites import get_suites
from agentdojo.agent_pipeline.agent_pipeline import AgentPipeline, PipelineConfig
from agentdojo.agent_pipeline.llms.openai_llm import OpenAILLM
from agentdojo.benchmark import benchmark_suite_without_injections
import openai, time


In [ ]:
import threading
from pathlib import Path

def _heartbeat(label, t0, stop, every=300):
    d = Path('/kaggle/working/runs') / label
    while not stop.wait(every):
        n = len(list(d.rglob('*.json'))) if d.is_dir() else 0
        print(f'[heartbeat] {label}: {time.time()-t0:.0f}s elapsed, {n} json traces so far', flush=True)

def run_utility(model_path, label, split='1,1'):
    from agentdojo.logging import OutputLogger
    print(f'{label}: loading weights / starting llama-server...', flush=True)
    proc = serve(model_path, split=split)
    stop = threading.Event()
    try:
        client = openai.OpenAI(api_key='EMPTY', base_url='http://127.0.0.1:8099/v1')
        mid = client.models.list().data[0].id
        print(f'{label}: model id={mid}', flush=True)
        llm = OpenAILLM(client, mid)
        pipe = AgentPipeline.from_config(PipelineConfig(llm=llm, model_id=mid,
               defense=None, system_message_name=None, system_message=None))
        pipe.name = label
        suites = get_suites('v1.2.2')
        results, ns, secs = {}, {}, {}
        t0 = time.time(); n_eps = 0
        threading.Thread(target=_heartbeat, args=(label, t0, stop), daemon=True).start()
        # Official layout: runs/{pipeline_name}/suite/... (D19). Do not nest label in logdir.
        logdir = Path('/kaggle/working/runs')
        logdir.mkdir(parents=True, exist_ok=True)
        with OutputLogger(str(logdir)):
            for s in ['banking','slack']:
                print(f'{label}: starting {s} utility (no attack)...', flush=True)
                t_suite = time.time()
                r = benchmark_suite_without_injections(pipe, suites[s],
                        logdir=logdir, force_rerun=False,
                        benchmark_version='v1.2.2')
                u = list(r['utility_results'].values())
                n_eps += len(u)
                results[s] = sum(u)/len(u) if u else float('nan')
                ns[s] = len(u); secs[s] = time.time()-t_suite
                print(f'{label}/{s}: utility={results[s]:.3f}, n={len(u)}, {secs[s]:.0f}s', flush=True)
        dt = time.time()-t0
        eph = n_eps/(dt/3600) if dt > 0 else 0.0
        print(f'{label}: utility {results}, episodes {n_eps}, {dt:.0f}s, {eph:.1f} episodes/h', flush=True)
        model_dir = logdir / label
        n_json = sum(1 for _ in model_dir.rglob('*.json')) if model_dir.is_dir() else 0
        print(f'{label}: traces under {model_dir}: {n_json} json files', flush=True)
        return {'model_id': mid, 'utility': results, 'n': ns, 'suite_seconds': secs,
                'episodes': n_eps, 'wall_s': dt, 'eps_per_h': eph}
    finally:
        stop.set()
        proc.terminate()
        print(f'{label}: server stopped', flush=True)


In [ ]:
# Gemma-4-26B-A4B QAT Q4_0, split across 2 GPUs
GEMMA_RES = run_utility(GEMMA, 'gemma-4-26b-a4b', split='1,1')


## 4. Tool-call parse-error rate
Strict check aligned with AgentDojo EmptyFunctionCall / tool-call parse failure markers (research-plan §11.8: replace a model if parse errors >5%). Does **not** flag the bare substring `"error"` (that false-positive flagged 37/37 in pilot Version #1 — D18). Scans JSON under `/kaggle/working/runs/{label}/`.

In [ ]:
import json, glob, re
from pathlib import Path

# Precise markers only. Do NOT match bare 'error' (D18).
PARSE_MARKERS = (
    'EmptyFunctionCall',
    'could not parse function call',
    'Invalid function call',
    'Invalid function calling output',
    'ASTParsingError',
    'Error parsing the model output',
    '<empty-function-name>',
    'could not parse',
)
_MARKER_RE = re.compile('|'.join(re.escape(m) for m in PARSE_MARKERS), re.IGNORECASE)

def _episode_has_parse_error(d):
    top = d.get('error')
    if isinstance(top, str) and top and _MARKER_RE.search(top):
        return True
    for msg in d.get('messages') or []:
        if not isinstance(msg, dict):
            continue
        err = msg.get('error')
        if isinstance(err, str) and err and _MARKER_RE.search(err):
            return True
        content = msg.get('content')
        chunks = []
        if isinstance(content, str):
            chunks.append(content)
        elif isinstance(content, list):
            for b in content:
                if isinstance(b, dict) and isinstance(b.get('content'), str):
                    chunks.append(b['content'])
                elif isinstance(b, str):
                    chunks.append(b)
        for c in chunks:
            if _MARKER_RE.search(c):
                return True
    return False

def parse_error_rate(label, threshold=0.05):
    files = sorted(glob.glob(f'/kaggle/working/runs/{label}/**/*.json', recursive=True))
    tot = err = 0
    flagged = []
    for fp in files:
        try:
            d = json.load(open(fp))
        except Exception:
            continue
        if not isinstance(d, dict) or 'messages' not in d:
            continue
        tot += 1
        if _episode_has_parse_error(d):
            err += 1
            flagged.append(fp)
    rate = err / tot if tot else float('nan')
    over = (rate > threshold) if tot else False
    print(f"{label}: parse-errors {err}/{tot} = {rate:.1%}  "
          f"({'EXCEEDS' if over else 'within'} §11.8 {threshold:.0%} replacement rule)")
    if flagged[:5]:
        print(f'  examples: {flagged[:5]}')
    return rate

PARSE_RATE = parse_error_rate('gemma-4-26b-a4b')


## 5. Trace inventory (keep JSON in Kaggle Output)
List `/kaggle/working/runs` so Save & Run All commits per-episode AgentDojo JSON (D19).

In [ ]:
import json, os
from pathlib import Path

runs = Path('/kaggle/working/runs')
manifest = {'runs_root': str(runs), 'models': {}}
print(f'runs root exists={runs.is_dir()}')
if runs.is_dir():
    for model_dir in sorted(p for p in runs.iterdir() if p.is_dir()):
        files = list(model_dir.rglob('*.json'))
        nbytes = sum(f.stat().st_size for f in files if f.is_file())
        manifest['models'][model_dir.name] = {
            'n_json': len(files), 'bytes': nbytes,
            'sample': [str(f.relative_to(runs)) for f in files[:5]],
        }
        print(f'  {model_dir.name}: {len(files)} json, {nbytes/1024:.1f} KiB')
        for f in files[:3]:
            print(f'    e.g. {f.relative_to(runs)}')
else:
    print('  WARNING: /kaggle/working/runs missing — traces will not appear in Output')

man_path = Path('/kaggle/working/runs_manifest.json')
man_path.write_text(json.dumps(manifest, indent=2))
print(f'wrote {man_path} ({man_path.stat().st_size} bytes)')
print('/kaggle/working top-level:')
for p in sorted(Path('/kaggle/working').iterdir()):
    if p.name.startswith('.'): continue
    print(f"  {'dir' if p.is_dir() else 'file'} {p.name}")


## 6. Result + keep/drop marker (D20)
Writes `/kaggle/working/gemma_pilot_result.json` and prints one `GEMMA_PILOT_RESULT` line to copy into DECISIONS.md.

In [ ]:
import json, math
KEEP_EPH = 64.0
u = GEMMA_RES['utility']
util_ok = all((v >= 0.20) for v in u.values())
parse_ok = (not math.isnan(PARSE_RATE)) and PARSE_RATE <= 0.05
eph_ok = GEMMA_RES['eps_per_h'] >= KEEP_EPH
verdict = 'KEEP' if (eph_ok and util_ok and parse_ok) else 'REVIEW'
out = dict(GEMMA_RES, parse_error_rate=PARSE_RATE, serve=SERVE_USED,
           gguf={'repo': GEMMA_REPO, 'rev': GEMMA_REV, 'file': GEMMA_FILE, 'sha256': GEMMA_SHA},
           llama_build=LLAMA_BUILD, keep_threshold_eps_per_h=KEEP_EPH,
           eph_ok=eph_ok, util_ok=util_ok, parse_ok=parse_ok, verdict=verdict)
open('/kaggle/working/gemma_pilot_result.json','w').write(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))
print(f"GEMMA_PILOT_RESULT eps_per_h={GEMMA_RES['eps_per_h']:.1f} "
      f"banking={u.get('banking', float('nan')):.3f} slack={u.get('slack', float('nan')):.3f} "
      f"parse={PARSE_RATE:.1%} variant={SERVE_USED['variant']} verdict={verdict}", flush=True)
